# Train a language model

You will train a decoder-only transformer with the parts current open models use: RMSNorm, rotary positions, grouped-query attention and a gated MLP. `TokenWindows` cuts a token stream into training windows, `LMObjective` owns the next-token loss, and `Trainer` owns gradients, the EMA, checkpoints and evaluation. By the end the model writes text from a prompt, you have restored its checkpoint from disk, and you have sampled from the restored weights.

Two routes share every cell but the configuration:

- **Offline route (default, runs here).** A small generated corpus of English-like sentences, byte-tokenized, and a two-layer model. It runs on a CPU in a few minutes and shows the model learning the corpus; it does not produce interesting text.
- **Tiny Shakespeare route.** `ROUTE = "shakespeare"` downloads the 1.1 MB corpus and trains a 12-layer, 512-wide model for 20 epochs, which takes a few minutes on a GPU or TPU. That route was not executed while writing this notebook.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml @ git+https://github.com/AshishKumar4/dew" {jax_spec}

In [ ]:
ROUTE = "offline"         # "offline" runs anywhere; "shakespeare" downloads the corpus and wants an accelerator
DATA_DIR = "data/05-tokens"
RUN_DIR = "runs/05-lm"
PROMPT = "the cat "

if ROUTE == "offline":
    SEQUENCE_LENGTH = 32
    BATCH_SIZE = 16
    STEPS = 300
    LEARNING_RATE = 3e-3
    MODEL = dict(emb_features=64, num_layers=2, num_heads=4, dropout_rate=0.0)
    DTYPE, ATTENTION = "float32", "xla"
    MAX_NEW_TOKENS = 48
else:
    SEQUENCE_LENGTH = 256
    BATCH_SIZE = 64
    STEPS = 1320              # 20 epochs of 66 steps on this corpus
    LEARNING_RATE = 1e-3
    MODEL = dict(emb_features=512, num_layers=12, num_heads=8, dropout_rate=0.2)
    DTYPE, ATTENTION = "bfloat16", "auto"
    MAX_NEW_TOKENS = 400
    PROMPT = "ROMEO:"
SEED = 0

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## The data

A language model predicts the next token, so the corpus becomes one flat stream of token ids and the loader cuts it into windows of `SEQUENCE_LENGTH + 1`: the first `SEQUENCE_LENGTH` ids of a row are inputs and the same ids shifted by one are the targets. Each window overlaps the previous one by exactly one token.

The token files are `train.bin`, `val.bin` and `meta.json`, the layout `tools/tokenize_text.py` writes from a text file. The cell below writes the same layout in the notebook with the byte tokenizer, whose vocabulary of 256 needs no download. The validation split is the head of the stream, as the tool splits it.

In [ ]:
import json
from pathlib import Path

import numpy as np
from dew.data.text import ByteTokenizer

data_dir = Path(DATA_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
raw = data_dir / "corpus.txt"

if ROUTE == "offline":
    rng = np.random.default_rng(SEED)
    subjects = ["the cat", "a dog", "the bird", "my friend", "the child"]
    verbs = ["sees", "likes", "finds", "wants", "hears"]
    objects = ["the ball", "a tree", "the river", "some food", "the moon"]
    lines = [f"{rng.choice(subjects)} {rng.choice(verbs)} {rng.choice(objects)}.\n"
             for _ in range(4000)]
    raw.write_text("".join(lines), encoding="utf-8")
else:
    import urllib.request
    if not raw.exists():
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
            raw)
print("raw text:", raw.stat().st_size, "bytes")

tokenizer = ByteTokenizer()
ids = np.asarray(tokenizer.encode(raw.read_text(encoding="utf-8")))
val_len = int(round(len(ids) * 0.02))
val, train = ids[:val_len], ids[val_len:]
val.astype(np.uint8).tofile(data_dir / "val.bin")
train.astype(np.uint8).tofile(data_dir / "train.bin")
meta = {"tokenizer": "byte", "vocab_size": tokenizer.vocab_size, "dtype": "uint8",
        "train_tokens": len(train), "val_tokens": len(val), "eos_id": None}
(data_dir / "meta.json").write_text(json.dumps(meta, indent=2))
print(meta)

In [ ]:
from dew.data import Loading, TokenWindows

data = TokenWindows(path=DATA_DIR, seq_len=SEQUENCE_LENGTH, val_batches=4,
                    loading=Loading(workers=0, threads=1, read_buffer=2)).load(batch=BATCH_SIZE)
print("training windows:", data.records, "| steps per epoch:", data.steps_per_epoch)

batch = next(iter(data.train()))
print(batch["text"].shape, batch["text"].dtype)
print(repr(tokenizer.decode(batch["text"][0][:40])))

## The model and the objective

`models.build("causal_transformer", ...)` constructs the decoder from the registry. `dtype` is the compute dtype and the parameters stay fp32; `attention_impl` picks the attention kernel. The vocabulary comes from `meta.json`, and `max_seq_len` has to cover the training context plus everything the final sample generates, because the KV cache is allocated once at that length.

`LMObjective` is next-token cross entropy: it shifts the batch itself, scores the logits in fp32, and reports the loss and token accuracy. `Samples` configures the text preview an evaluation writes, from a fixed prompt with the given `Sampling`; `metrics.perplexity()` scores every validation window.

In [ ]:
from dew import models
from dew.objectives.lm import LMObjective, Samples
from dew.sampling import Sampling

prompt_ids = tokenizer.encode(PROMPT)
model = models.build(
    "causal_transformer", vocab_size=meta["vocab_size"], **MODEL,
    max_seq_len=max(SEQUENCE_LENGTH, len(prompt_ids) + MAX_NEW_TOKENS),
    dtype=DTYPE, attention_impl=ATTENTION)
objective = LMObjective(
    model, SEQUENCE_LENGTH, ema_decay=0.99,
    samples=Samples(prompt=prompt_ids, max_new_tokens=MAX_NEW_TOKENS,
                    sampling=Sampling(temperature=0.8, top_k=40), decode=tokenizer.decode))

variables = jax.eval_shape(objective.init, jax.random.key(0))
print(f"{sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(variables)) / 1e6:.2f}M parameters")

## Training

`Trainer` compiles the objective's loss into one step with the optimizer update and the EMA average inside it, and writes checkpoints under `RUN_DIR`. `eval_every` schedules the validation pass, which scores the held-out windows with `perplexity` from the EMA weights; the text preview goes to a tracker when one is configured. The learning rate warms up and then decays with a cosine over the run.

In [ ]:
import optax
from dew import Checkpoints, Trainer, metrics

schedule = optax.warmup_cosine_decay_schedule(
    init_value=LEARNING_RATE * 0.01, peak_value=LEARNING_RATE,
    warmup_steps=max(1, STEPS // 10), decay_steps=STEPS, end_value=LEARNING_RATE * 0.1)
trainer = Trainer(objective, optax.adamw(schedule), key=jax.random.key(SEED),
                  checkpoints=Checkpoints(RUN_DIR))
state = trainer.fit(data, steps=STEPS, log_every=max(1, STEPS // 6),
                    eval_every=max(1, STEPS // 3), checkpoint_every=STEPS,
                    metrics=(metrics.perplexity(),))
print("optimizer updates:", int(state.updates))

## Perplexity

Perplexity is `exp(cross entropy)`: on average, how many tokens the model was choosing between at each position. A byte-level model that has just started scores near 256, and the offline corpus is regular enough that it falls into single digits. What matters more than the absolute number is the gap between training and validation, which grows when a model memorises a corpus that is too small for it.

The trainer already reported `val/perplexity` from the whole validation pass. The cell below rescores the same windows through the objective's own evaluation call, from the EMA weights, so you can see the pieces: `TokenScores` carries a per-target loss and a weight, and perplexity is `exp` of their weighted mean.

In [ ]:
from dew.objectives.base import Step

losses, weights = [], []
for batch in data.val():
    scores = objective.evaluate(state.params, batch,
                                Step(step=state.step, key=jax.random.key(1), ema=state.averaged))
    losses.append(np.asarray(scores.losses))
    weights.append(np.asarray(scores.weights))
losses, weights = np.concatenate(losses), np.concatenate(weights)
cross_entropy = float((losses * weights).sum() / weights.sum())
print(f"validation cross entropy {cross_entropy:.4f}  perplexity {np.exp(cross_entropy):.2f}")

## Generation and the KV cache

A decoder generates one token at a time, and each new token attends to every token before it. The KV cache keeps the keys and values of every position in a buffer sized `max_seq_len`: the prompt runs through the model once to fill it, and every later step runs a single token.

`generate` runs that loop compiled. `Sampling` carries temperature, top-k and an optional EOS id; the `Generation` it returns keeps the prompt in `tokens`, counts each row's response `lengths`, and records the log-probabilities of the sampled tokens. Temperature 0 is greedy decoding.

In [ ]:
import jax.numpy as jnp
from dew.sampling import generate

prompt = jnp.asarray([prompt_ids], jnp.int32)
greedy = generate(model, state.averaged, prompt, max_new_tokens=MAX_NEW_TOKENS,
                  key=jax.random.key(0), sampling=Sampling(temperature=0.0))
print("greedy:", repr(tokenizer.decode(greedy.tokens[0])))
print("response length:", int(greedy.lengths[0]), "| terminated by EOS:", bool(greedy.terminated[0]))

sampled = generate(model, state.averaged, prompt, max_new_tokens=MAX_NEW_TOKENS,
                   key=jax.random.key(1), sampling=Sampling(temperature=0.8, top_k=40))
print("temperature 0.8, top-k 40:", repr(tokenizer.decode(sampled.tokens[0])))

## Checkpoints

`fit` wrote a checkpoint at the end of the run under `RUN_DIR`. It holds the parameters, the EMA, the optimizer state, the step clocks, the run key and the data position. A new `Trainer` over the same objective and optimizer restores it with `place()`; a checkpoint written on one mesh loads onto another. Sampling from the restored EMA weights with the greedy setting reproduces the text above.

In [ ]:
restored_trainer = Trainer(objective, optax.adamw(schedule), key=jax.random.key(SEED),
                           checkpoints=Checkpoints(RUN_DIR))
restored, _, _ = restored_trainer.place()
print("restored at step", int(restored.step))
again = generate(model, restored.averaged, prompt, max_new_tokens=MAX_NEW_TOKENS,
                 key=jax.random.key(0), sampling=Sampling(temperature=0.0))
print("same greedy text after restore:", bool(np.array_equal(again.tokens, greedy.tokens)))

## Where to go next

**More text.** `tools/tokenize_text.py --input <file or directory of .txt> --out <dir> --tokenizer byte` writes the same three files for any corpus; a Hugging Face tokenizer name packs the text into far fewer tokens at the cost of a larger embedding table. The same run from the command line is `python recipes/lm/train.py data:token-windows --data.path <dir> --data.seq-len 256 --model.config '{"emb_features": 512, "num_layers": 12, "num_heads": 8}'`.

[Notebook 07](07-scaling-on-many-devices.ipynb) trains this objective on several devices at once, and [notebook 08](08-load-a-pretrained-decoder.ipynb) starts from a pretrained Hugging Face decoder instead of random weights.